In [2]:
import pandas as pd
import numpy as np
import time

print("Avvio Modulo 02: Data Ingestion & Vettorial Merge...\n")
start_time = time.time()

# --- 1. CONFIGURAZIONE PERCORSI ---
path_es_03 = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 03-26.Last.txt"
path_es_06 = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 06-26.Last.txt"
path_cfd = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\fpmarkets_us500_ticks.csv"
path_output = r"C:\Users\Leona\source\repos\QuantEdge_Project\data\processed\es_master_4months.parquet"

# --- 2. GESTIONE FUSI ORARI E ALLINEAMENTO ---
# IMPOTANTE: Se NinjaTrader era impostato sul tuo orologio locale italiano, 
# cambia 'US/Central' con 'Europe/Rome'. Altrimenti lascia l'orario di Chicago.
nt_timezone = 'US/Central'

# Il file CFD inizia il 9 Febbraio 2026. Tagliamo il Future in eccesso.
data_inizio_cut = pd.to_datetime('2026-02-09 01:00:00').tz_localize('UTC').tz_convert(nt_timezone).tz_localize(None)

# --- 3. CARICAMENTO E PRE-PROCESSING FUTURE ---
def processa_future(file_path, start_date=None):
    print(f"Lettura e calcolo Delta per: {file_path}")
    df = pd.read_csv(
        file_path, sep=';', 
        names=['RawTime', 'Price', 'Bid', 'Ask', 'Volume'],
        usecols=[0, 1, 2, 3, 4], engine='c'
    )
    
    # Conversione temporale ottimizzata
    df['CleanTime'] = df['RawTime'].str.slice(0, 15)
    df['Datetime'] = pd.to_datetime(df['CleanTime'], format='%Y%m%d %H%M%S')
    
    # Taglio chirurgico per risparmiare memoria (applicato solo al file di Marzo)
    if start_date:
        df = df[df['Datetime'] >= start_date]
        
    # Conversione in UTC Assoluto per il merge
    df['Datetime_UTC'] = df['Datetime'].dt.tz_localize(nt_timezone).dt.tz_convert('UTC')
    
    # Calcolo del Delta Istantaneo (Tick Rule avanzata)
    df['Direction'] = np.where(df['Price'] >= df['Ask'], 1, np.where(df['Price'] <= df['Bid'], -1, 0))
    df['Delta'] = df['Volume'] * df['Direction']
    
    # Pulizia colonne: teniamo solo l'essenziale per il quant trading
    df = df[['Datetime_UTC', 'Price', 'Volume', 'Delta']].sort_values('Datetime_UTC')
    return df

df_03 = processa_future(path_es_03, start_date=data_inizio_cut)
df_06 = processa_future(path_es_06)

print("Concatenazione contratti Future...")
df_future = pd.concat([df_03, df_06], ignore_index=True)
del df_03, df_06 # Svuotamento forzato della RAM

# --- 4. CARICAMENTO DATI CFD ---
print("Lettura storico CFD FP Markets...")
df_cfd = pd.read_csv(path_cfd, sep=';', usecols=['datetime_utc', 'bid', 'ask'], engine='c')
df_cfd['Datetime_UTC'] = pd.to_datetime(df_cfd['datetime_utc'], format='ISO8601')
df_cfd = df_cfd[['Datetime_UTC', 'bid', 'ask']].sort_values('Datetime_UTC')

# --- 5. IL MERGE VETTORIALE (ALLINEAMENTO MILLISECONDI) ---
print("Inizio Fusione Asincrona (Merge Asof)...")
# Abbiniamo ogni tick del Future al prezzo Bid/Ask del CFD più vicino nel futuro
df_master = pd.merge_asof(
    df_future, 
    df_cfd, 
    on='Datetime_UTC', 
    direction='forward',
    tolerance=pd.Timedelta(seconds=2) # Evita abbinamenti errati se ci sono buchi
)

# --- 6. SALVATAGGIO COMPRESSO ---
print(f"Salvataggio del Master Dataset in formato Parquet: {path_output}")
df_master.to_parquet(path_output, index=False)

end_time = time.time()
print(f"\n--- FUSIONE COMPLETATA IN {(end_time - start_time):.2f} SECONDI ---")
print(f"Righe Totali Master: {len(df_master):,}")
print("Il dataset è pronto per lo sviluppo dell'algoritmo.")

Avvio Modulo 02: Data Ingestion & Vettorial Merge...

Lettura e calcolo Delta per: C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 03-26.Last.txt
Lettura e calcolo Delta per: C:\Users\Leona\source\repos\QuantEdge_Project\data\raw\ES 06-26.Last.txt
Concatenazione contratti Future...
Lettura storico CFD FP Markets...
Inizio Fusione Asincrona (Merge Asof)...
Salvataggio del Master Dataset in formato Parquet: C:\Users\Leona\source\repos\QuantEdge_Project\data\processed\es_master_4months.parquet


ArrowKeyError: No type extension with name arrow.py_extension_type found

In [3]:
# 1. Rimuoviamo il metadato problematico del fuso orario (il dato resta identico, perde solo l'etichetta)
df_master['Datetime_UTC'] = df_master['Datetime_UTC'].dt.tz_localize(None)

# 2. Salviamo forzando il motore 'fastparquet'
print(f"Salvataggio in corso in: {path_output}")
df_master.to_parquet(path_output, index=False, engine='fastparquet')

print("File Parquet salvato con successo. Infrastruttura completata.")

Salvataggio in corso in: C:\Users\Leona\source\repos\QuantEdge_Project\data\processed\es_master_4months.parquet
File Parquet salvato con successo. Infrastruttura completata.
